# One sensor (milestone 2)

This notebook replays an hour of truth past a simulated rotating 2D
surveillance radar and checks that the radar behaves as configured:

- detections plot over truth, with the errors a real 2D radar has (noise,
  slant range, unresolved targets, false alarms);
- the achieved probability of detection matches the configured Pd within
  sampling error;
- two runs with one seed are identical.

It runs as-is on the synthetic fixture and the example radar at Rocky
Mountain Metropolitan Airport. To use your own data, set these before
starting Jupyter (or edit the next cell):

| Variable | Meaning |
| --- | --- |
| `TRACKING_SIM_RECORDING` | A recording or SBS capture (see notebook 01) |
| `TRACKING_SIM_SBS_TZ` | Time zone of SBS timestamps, if the recording is SBS |
| `TRACKING_SIM_ORIGIN` | `LAT,LON[,ALT_M]` of the scenario frame (default: the radar site) |
| `TRACKING_SIM_SENSOR` | A sensor spec YAML (default `scenarios/sensors/radar-1.yaml`) |
| `TRACKING_SIM_SEED` | The run seed (default 42) |

In [ ]:
import itertools
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow.compute as pc

from tracking_sim.geo import LocalFrame
from tracking_sim.sensors import load_sensor_spec
from tracking_sim.sim import ReplayEngine, wilson_interval
from tracking_sim.truth import build_truth
from tracking_sim.truth.sources.recording import RecordingSource
from tracking_sim.truth.sources.sbs import SbsSource

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())


def from_root(name):
    # A path from an environment variable; relative paths are from the repository root.
    value = os.environ.get(name)
    return str(ROOT / value) if value else None


RECORDING = from_root("TRACKING_SIM_RECORDING") or str(ROOT / "tests/data/synthetic_1h.parquet")
SENSOR = from_root("TRACKING_SIM_SENSOR") or str(ROOT / "scenarios/sensors/radar-1.yaml")
SEED = int(os.environ.get("TRACKING_SIM_SEED", "42"))
SBS_TZ = os.environ.get("TRACKING_SIM_SBS_TZ", "UTC")

# Colours: the reference data-viz palette's first two categorical slots, plus neutrals.
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, GREY, LIGHT = "#52514e", "#8a8984", "#c9c8c2"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("recording:", RECORDING)
print("sensor:   ", SENSOR)

## 1. Truth and the radar

The scenario frame is centred on the radar unless `TRACKING_SIM_ORIGIN` says
otherwise. The radar is a YAML spec; this one turns every 5 s, measures slant
range (60 m noise) and bearing (0.2 degrees), detects with Pd 0.9 out to
110 km, and sees about 38 false alarms a scan.

In [ ]:
spec = load_sensor_spec(SENSOR)
origin = os.environ.get("TRACKING_SIM_ORIGIN")
if origin:
    frame = LocalFrame(*[float(x) for x in origin.split(",")])
else:
    frame = LocalFrame(spec.position.lat, spec.position.lon, spec.position.alt_m)

if ".sbs" in Path(RECORDING).suffixes:
    source = SbsSource(path=RECORDING, receiver="primary", tz=SBS_TZ)
else:
    source = RecordingSource(RECORDING)
truth = build_truth(list(source.replay()), frame)
print(f"{len(truth)} truth segments, {truth.stats.aircraft} aircraft")
print(spec.model_dump_json(indent=2, exclude_defaults=True))

## 2. Run it

`ReplayEngine` finds when the beam crosses each aircraft, samples truth at
that moment, applies the range, elevation and radar-horizon gates, draws
detections with Pd, adds noise, merges targets it can't resolve and adds
false alarms. Everything random comes from the seed.

In [ ]:
engine = ReplayEngine(truth, [spec], seed=SEED)
result = engine.run()
detections = result.detections_table()
opportunities = result.opportunities_table()
stats = result.stats(spec.id)
print(result.summary())

targets = detections.filter(pc.equal(detections["kind"], "target")).to_pandas()
clutter = detections.filter(pc.equal(detections["kind"], "clutter")).to_pandas()
opps = opportunities.to_pandas()

## 3. Detections over truth

The left panel is a PPI view, the way a 2D radar display draws the world:
each detection sits at its measured slant range along its measured bearing.
Truth (grey lines) is drawn at its true ground position. Clutter is sampled
down to one scan in ten so the targets stay visible.

The right panel follows one aircraft. A 2D radar measures *slant* range, so
a high aircraft shows up farther out than it is along the ground; the
detections track the true slant range, not the ground range.

In [ ]:
fig, (ax, ax_r) = plt.subplots(1, 2, figsize=(14, 6.5), gridspec_kw={"width_ratios": [1.15, 1]})
some_clutter = clutter[clutter["scan"] % 10 == 0]
ax.scatter(
    some_clutter["east_m"] / 1e3,
    some_clutter["north_m"] / 1e3,
    s=2,
    color=LIGHT,
    lw=0,
    label=f"false alarms (1 scan in 10, {stats.false_alarms_per_scan:.0f} per scan)",
)
for seg in truth:
    t = np.arange(seg.t[0], seg.t[-1], 5.0)
    pos, _, ok = seg.evaluate(t)
    ax.plot(pos[ok, 0] / 1e3, pos[ok, 1] / 1e3, color=GREY, lw=0.8)
ax.scatter(
    targets["east_m"] / 1e3,
    targets["north_m"] / 1e3,
    s=4,
    color=BLUE,
    lw=0,
    label=f"target detections ({len(targets)})",
)
radar_xy = result.radars[spec.id].site.position / 1e3
for r_km in (25, 50, 75, 100):
    ax.add_patch(plt.Circle(radar_xy[:2], r_km, fill=False, ls=":", color=GREY, lw=0.6))
ax.add_patch(
    plt.Circle(radar_xy[:2], spec.coverage.range_max_m / 1e3, fill=False, color=INK, lw=0.8)
)
ax.plot(*radar_xy[:2], "^", color=INK, ms=7)
ax.plot([], [], color=GREY, lw=0.8, label="truth")
ax.set(xlabel="east (km)", ylabel="north (km)", title=f"{spec.id}: one hour, PPI view")
ax.set_aspect("equal")
ax.legend(loc="upper left", fontsize=8, frameon=False)

seg_id = (
    targets.groupby("segment_id")["true_elevation_deg"]
    .mean()
    .loc[lambda s: targets.groupby("segment_id").size()[s.index] > 100]
    .idxmax()
)
mine = targets[targets["segment_id"] == seg_id]
seen = opps[(opps["segment_id"] == seg_id) & opps["in_coverage"]]
t0 = seen["timestamp"].min()


def minutes(ts):
    return (ts - t0).dt.total_seconds() / 60


ax_r.plot(
    minutes(seen["timestamp"]),
    seen["ground_range_m"] / 1e3,
    color=GREY,
    lw=1.2,
    ls="--",
    label="true ground range",
)
ax_r.plot(
    minutes(seen["timestamp"]),
    seen["slant_range_m"] / 1e3,
    color=INK,
    lw=1.2,
    label="true slant range",
)
ax_r.scatter(
    minutes(mine["timestamp"]),
    mine["slant_range_m"] / 1e3,
    s=8,
    color=BLUE,
    lw=0,
    label="measured slant range",
)
missed = seen[~seen["detected"]]
ax_r.scatter(
    minutes(missed["timestamp"]),
    missed["slant_range_m"] / 1e3,
    s=18,
    marker="x",
    color=ORANGE,
    lw=1.2,
    label=f"missed ({len(missed)})",
)
ax_r.set(xlabel="minutes", ylabel="range (km)", title=f"{seg_id}: slant vs ground range")
ax_r.legend(fontsize=8, frameon=False)
fig.tight_layout()

## 4. Measurement errors

Range and bearing errors of every target detection that came from a single
aircraft, against the configured noise (dashed). Merged detections are left
out: their error includes the spacing between the aircraft.

In [ ]:
single = targets[targets["merged"] == 1]
dr = single["slant_range_m"] - single["true_slant_range_m"]
db = (single["azimuth_deg"] - single["true_azimuth_deg"] + 180) % 360 - 180
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, err, sigma, unit in (
    (a1, dr, spec.noise.range_sigma_m, "m"),
    (a2, db, spec.noise.bearing_sigma_deg, "deg"),
):
    ax.hist(err, bins=60, density=True, color=BLUE, alpha=0.85, rwidth=0.9)
    x = np.linspace(-4 * sigma, 4 * sigma, 200)
    ax.plot(
        x,
        np.exp(-0.5 * (x / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi)),
        color=INK,
        ls="--",
        lw=1.2,
    )
    ax.set(
        xlabel=f"error ({unit})",
        yticks=[],
        title=f"std {err.std():.3g} {unit} (configured {sigma:g})",
    )
a1.set_ylabel("density")
fig.tight_layout()
print(f"{len(targets) - len(single)} merged detections left out")

## 5. Achieved Pd

Pd counts every beam crossing that passed the geometry gates (an
*opportunity*), so the achieved Pd should match the configured value at
every range, within sampling error. Bars are 95% Wilson intervals.

In [ ]:
inside = opps[opps["in_coverage"]]
edges = np.linspace(0, spec.coverage.range_max_m, 12)
centre, pd, lo, hi, n = [], [], [], [], []
for a, b in itertools.pairwise(edges):
    sel = inside[(inside["slant_range_m"] >= a) & (inside["slant_range_m"] < b)]
    if len(sel) < 20:
        continue
    k = int(sel["detected"].sum())
    low, high = wilson_interval(k, len(sel))
    centre.append((a + b) / 2e3)
    pd.append(k / len(sel))
    lo.append(low)
    hi.append(high)
    n.append(len(sel))
pd, lo, hi = map(np.array, (pd, lo, hi))
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.axhline(
    spec.detection.pd, color=INK, ls="--", lw=1.2, label=f"configured Pd {spec.detection.pd}"
)
ax.errorbar(
    centre,
    pd,
    yerr=[pd - lo, hi - pd],
    fmt="o",
    ms=5,
    color=BLUE,
    capsize=3,
    lw=1.2,
    label="achieved Pd (95% interval)",
)
ax.set(xlabel="slant range (km)", ylabel="Pd", ylim=(0.5, 1.02), title="Achieved Pd by range")
ax.legend(fontsize=8, frameon=False, loc="lower left")
fig.tight_layout()
lo_all, hi_all = stats.pd_interval
print(
    f"overall: achieved Pd {stats.achieved_pd:.3f}, 95% interval [{lo_all:.3f}, {hi_all:.3f}], "
    f"configured {stats.configured_pd}: {'consistent' if stats.pd_consistent else 'NOT consistent'}"
)

## 6. False alarms

The number of false alarms per scan is Poisson with mean
`false_alarm_density × coverage area`; the dots are that distribution.

In [ ]:
from math import exp, lgamma, log

per_scan = clutter.groupby("scan").size().reindex(range(stats.scans), fill_value=0)
lam = stats.expected_false_alarms_per_scan
k = np.arange(per_scan.min(), per_scan.max() + 1)
poisson = np.array([exp(x * log(lam) - lam - lgamma(x + 1)) for x in k]) * stats.scans
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(
    per_scan,
    bins=np.arange(k[0] - 0.5, k[-1] + 1.5),
    color=BLUE,
    alpha=0.85,
    rwidth=0.85,
    label="scans",
)
ax.plot(k, poisson, "o", color=INK, ms=4, label=f"Poisson, mean {lam:.1f}")
ax.set(xlabel="false alarms in a scan", ylabel="scans", title="False alarms per scan")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()

## 7. Same seed, same run

Re-running with the same seed reproduces every detection exactly; another
seed changes the draws but not the geometry (the same beam crossings).

In [ ]:
again = ReplayEngine(truth, [spec], seed=SEED).run()
other = ReplayEngine(truth, [spec], seed=SEED + 1).run()
same = again.detections_table().equals(detections) and again.opportunities_table().equals(
    opportunities
)
print(f"seed {SEED} twice: identical = {same}")
print(
    f"seed {SEED + 1}: identical = {other.detections_table().equals(detections)}, "
    f"{len(other.detections())} detections vs {len(result.detections())}"
)
print(
    "same beam crossings:",
    other.opportunities_table().drop(["detected"]).equals(opportunities.drop(["detected"])),
)

## 8. What a tracker receives

Detections are Stone Soup objects: `TrueDetection` for aircraft (linked to
the truth path, for scoring) and `Clutter` for false alarms, each carrying
the slant-range measurement model a tracker needs. Measurements are
`[bearing, slant range]` with Stone Soup's bearing convention
(counter-clockwise from east). Milestone 3 feeds these to trackers.

In [ ]:
dets = result.detections()
first_target = next(d for d in dets if d.metadata["kind"] == "target")
first_clutter = next(d for d in dets if d.metadata["kind"] == "clutter")
for d in (first_target, first_clutter):
    print(type(d).__name__, d.timestamp, np.asarray(d.state_vector, dtype=float).ravel().round(4))
    print(
        "  ",
        {
            k: v
            for k, v in d.metadata.items()
            if k in ("sensor_id", "scan", "segment_id", "report_time")
        },
    )
print(type(first_target.measurement_model).__name__)
print(first_target.measurement_model.covar())